In [ ]:
import shap
import torch
import torch.nn as nn
import numpy as np

import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

import seaborn as sns

from sklearn.cluster import KMeans
import torch

from sklearn.decomposition import PCA
import networkx as nx
from torch_geometric.utils import to_networkx

from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.loader import DataLoader

from torch_geometric.explain.config import ModelConfig
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch3_data(filename="Feature_MLP.txt"):
    """
    Loads 17-dimensional numeric features from file.
    Each line has 17 floats (space- or comma-delimited).
    """
    X_branch3 = []
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 17 numbers per line
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)
    
    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 17)
    X_branch3 = X_branch3.reshape(len(X_branch3), 17)
    return X_branch3

def load_reaction_rates():
    with open('HTCas9_indel_frequency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)
    
########################################
# 2. Graph Data Utilities (for GNN)
########################################

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

########################################
# 2. MLP Branch
########################################

class MLPBranch17(nn.Module):
    """
    A single hidden layer MLP for 17D => output dimension mlp_dim.
    No separate "output" layer; just one fc + ReLU => final embedding.
    """
    def __init__(self, input_dim=17, hidden_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
    def forward(self, x):
        # x shape: (batch,17)
        x = F.relu(self.fc(x))  # (batch,hidden_dim)
        return x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


########################################
# 3. Final Fusion Model
########################################

class MLP_Only(nn.Module):
    """
    End-to-end: 
      - MLPBranch => feat_mlp
    dropout => final FC => 1
    """
    def __init__(self,
                 mlp_hidden_dim,  # MLP
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.mlp_branch = MLPBranch17(input_dim=17, hidden_dim=mlp_hidden_dim)
        
        # total dimension = (mlp_hidden_dim)
        total_dim = mlp_hidden_dim
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)


    def forward(self, x3):
        feat_mlp = self.mlp_branch(x3)                   # (batch, mlp_hidden_dim)

        feat_mlp = F.dropout(feat_mlp, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(feat_mlp))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)

########################################
# 4. Hybrid Dataset
########################################

class IndivDataset(Dataset):
    def __init__(self, X3, reaction_rates):
        super().__init__()
        self.X3 = X3
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(X3) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        x3 = torch.tensor(self.X3[idx], dtype=torch.float) 
        y = torch.tensor(self.reaction_rates[idx], dtype=torch.float)
        return x3, y

def collate(batch):
    from torch_geometric.data import Batch
    x3_list, y_list = zip(*batch)
    x3 = torch.stack(x3_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x3, y

class FullModelWrapperForMLP(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.full_model = full_model.eval()

    def forward(self, x3):
        batch_size = x3.shape[0]

        output = self.full_model(x3)

        return output.view(-1, 1)  # Required shape for SHAP


import torch
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import seaborn as sns
import numpy as np
import pandas as pd


np.random.seed(42)
torch.manual_seed(42)


# === Example Execution ===
if __name__ == "__main__":
    # Load your trained model
    full_model = torch.load(
    "MLP_trial_2.pt",
    weights_only=False
)
    full_model.eval()

    from sklearn.cluster import KMeans
    from sklearn.metrics import pairwise_distances_argmin_min
    import torch
    import numpy as np
    
    # Load original data
    X3_all = load_branch3_data()
    rates = load_reaction_rates()
    
    # Flatten for clustering
    X3_flat = X3_all.reshape(X3_all.shape[0], -1)
    
    # Run KMeans
    kmeans3_sample = KMeans(n_clusters=1000, random_state=42).fit(X3_flat)
    kmeans3_background = KMeans(n_clusters=100, random_state=42).fit(X3_flat)
    
    # Get indices of closest original points to each centroid
    closest_X3_indices_sample, _ = pairwise_distances_argmin_min(kmeans3_sample.cluster_centers_, X3_flat)
    closest_X3_indices_background, _ = pairwise_distances_argmin_min(kmeans3_background.cluster_centers_, X3_flat)
    
    # Get samples 
    X3_selected = X3_all[closest_X3_indices_sample]

    # Get backgrounds
    background_mlp = X3_all[closest_X3_indices_background]
    
    # Use these indices to extract the corresponding reaction rates
    rates_X3 = rates[closest_X3_indices_sample]

    # Fixed samples in different branches (sample is the one with median reaction rate)
    sample_idx = np.argsort(rates)[len(rates) // 2]  # index of median value
    x3_sample = torch.tensor(X3_all[sample_idx], dtype=torch.float).unsqueeze(0) 

    # === MLP SHAP ===
    
    # 1. Create the wrapper
    wrapper = FullModelWrapperForMLP(
        full_model=full_model)
    
    # 2. Build background and input tensors
    background_mlp = torch.tensor(background_mlp, dtype=torch.float)  
    X3_tensor = torch.tensor(X3_selected, dtype=torch.float)      
    X3_all_tensor = torch.tensor(X3_all, dtype=torch.float) 
    
    # 3. Run SHAP
    explainer = shap.GradientExplainer(wrapper, background_mlp)
    shap_values_mlp = explainer.shap_values(X3_tensor)
    shap_values_mlp = shap_values_mlp.squeeze(-1) 

    feature_names_17D = [
        "crRNA spacer",
        "bh_maximal consecutive paired bases",
        "bh_maximal consecutive unpaired bases",
        "bh_5' overhang",
        "bh_3' overhang",
        "bh_5' stem",
        "bh_3' stem",
        "dsDNA target",
        "ah_maximal consecutive paired bases",
        "ah_maximal consecutive unpaired bases",
        "ah_5' overhang",
        "ah_3' overhang",
        "ah_5' stem",
        "ah_3' stem",
        "PAM-proximal region",
        "central region",
        "PAM-distal region"
    ]

    import pandas as pd
    X_values = X3_tensor.cpu().numpy() if hasattr(X3_tensor, "cpu") else X3_tensor
    
    df_input = pd.DataFrame(X_values, columns=feature_names_17D)
    df_shap = pd.DataFrame(shap_values_mlp, columns=feature_names_17D)

    df_combined = pd.concat([df_input, df_shap], axis=1)
    
    df_combined.to_csv("indiv_shap_values_mlp_Cas9_2.csv", index=False)
    
    wrapper.eval()

print('finished')

In [ ]:
# Models tested: 2, 35, 41, 43, 45, 63, 73, 78, 79, 84

In [ ]:
import pandas as pd
import csv
from scipy.stats import spearmanr
from collections import Counter

# --- Config ---
file_names = [
    "indiv_shap_values_mlp_Cas9_2.csv", "indiv_shap_values_mlp_Cas9_35.csv",
    "indiv_shap_values_mlp_Cas9_42.csv", "indiv_shap_values_mlp_Cas9_43.csv",
    "indiv_shap_values_mlp_Cas9_45.csv", "indiv_shap_values_mlp_Cas9_63.csv",
    "indiv_shap_values_mlp_Cas9_73.csv", "indiv_shap_values_mlp_Cas9_78.csv",
    "indiv_shap_values_mlp_Cas9_79.csv", "indiv_shap_values_mlp_Cas9_84.csv"
]

results = []

# Step 1: collect results
for file_name in file_names:
    model_id = file_name.split("_")[-1].replace(".csv", "")  # Extract model ID like '27'

    with open(file_name) as f:
        reader = csv.reader(f)
        header = next(reader)

    df = pd.read_csv(file_name, header=None, skiprows=1)
    df.columns = header  # overwrite with true header

    col_counts = Counter(df.columns)
    duplicate_features = [col for col, count in col_counts.items() if count > 1]

    for feature in duplicate_features:
        indices = [i for i, col in enumerate(df.columns) if col == feature]
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                col1 = df.iloc[:, indices[i]]
                col2 = df.iloc[:, indices[j]]
                corr, pval = spearmanr(col1, col2)
                results.append({
                    "Feature": feature,
                    "Model": model_id,
                    "Spearman Correlation": corr,
                    "P-Value": pval
                })

# Step 2: Build DataFrame and order features
df = pd.DataFrame(results)

# Track first appearance order of features
feature_order = []
seen = set()
for row in results:
    feat = row["Feature"]
    if feat not in seen:
        seen.add(feat)
        feature_order.append(feat)

# Use category sorting to preserve order
df["Feature_order"] = pd.Categorical(df["Feature"], categories=feature_order, ordered=True)
df = df.sort_values(by=["Feature_order", "Model"])
df = df.drop(columns=["Feature_order"])

# Step 3: Blank out repeated feature names
def clear_repeated_features(df):
    cleared = []
    current_feat = None
    for _, row in df.iterrows():
        feat = row["Feature"]
        if feat == current_feat:
            row["Feature"] = ""
        else:
            current_feat = feat
        cleared.append(row)
    return pd.DataFrame(cleared)

df_clean = clear_repeated_features(df)

# Step 4: Save CSV
df_clean.to_csv("indiv_spearman_duplicates_grouped_norepeat_Cas9_mlp.csv", index=False)
print("Saved with feature names grouped and de-duplicated to 'indiv_spearman_duplicates_grouped_norepeat_Cas9_mlp.csv'")
